# 🚀 BGE-M3 Two-Stage Multi-Head LoRA Fine-Tuning
**Goal**: Fine-tune BGE-M3 (560M XLM-RoBERTa bidirectional encoder) on LLM author discrimination.
- **Stage 1**: Multi-Corpus Pretraining on Master Pool V2 (166k rows) with Family-Margin Cross-Entropy & Label Smoothing.
- **Stage 2**: Prompt-Grouped Contrastive Fine-Tuning on train.csv with Within-Prompt Centered Twin Head & SupCon Loss.
- **Validation**: Evaluated against the Clean 1,800-Row Unseen Benchmark.
- **Export**: Self-contained `heads.safetensors` and PEFT LoRA adapter for immediate drop-in integration.


In [ ]:
# ==============================================================================
# CELL 1: Environment, Reproducibility & Kaggle Torchao Neutralization
# ==============================================================================
import os
import gc
import json
import math
import time
import random
import importlib.util
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Iterator

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Neutralize Kaggle's incompatible pre-installed torchao (<0.16.0) across all libraries
_orig_find_spec = importlib.util.find_spec
def _safe_find_spec(name, *args, **kwargs):
    if name == 'torchao':
        return None
    return _orig_find_spec(name, *args, **kwargs)
importlib.util.find_spec = _safe_find_spec

try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, log_loss, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from safetensors.torch import save_file

# Reproducibility
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'[Env] PyTorch {torch.__version__} | Device: {device} | Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


In [ ]:
# ==============================================================================
# CELL 2: Configuration & Target Classes
# ==============================================================================
LABELS: List[str] = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
K = len(LABELS)
LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}

FAMILIES: Dict[str, List[int]] = {
    'gpt': [0, 1, 2, 3, 4],
    'gemini': [5, 6, 7, 8],
    'claude': [9, 10, 11, 12, 13],
    'glm': [15, 16]
}

CONFIG = {'version': '1.0.0', 'name': 'bge_m3_lora_training', 'description': 'Two-Stage Multi-Head LoRA Fine-Tuning for BGE-M3 (560M XLM-RoBERTa Bidirectional Encoder)', 'backbone': 'BAAI/bge-m3', 'max_len': 512, 'hidden_size': 1024, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'lora_target_modules': ['query', 'key', 'value', 'dense'], 'family_margin': 0.35, 'label_smoothing': 0.05, 'weight_decay': 0.01, 'stage1': {'enabled': True, 'epochs': 1, 'batch_size': 16, 'gradient_accumulation_steps': 2, 'lr_lora': 0.00015, 'lr_head': 0.0008, 'max_samples': 36000}, 'stage2': {'enabled': True, 'epochs': 2, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'lr_lora': 3e-05, 'lr_head': 0.0002, 'centered_weight': 0.65, 'supcon_weight': 0.2}, 'output_dir': '/kaggle/working/export_bge_m3_lora'}
print('[Config] Loaded training configuration for BGE-M3.')


In [ ]:
# ==============================================================================
# CELL 3: Multi-Source Data Discovery & Pipeline Assembly
# ==============================================================================
import hashlib

def find_input_file(patterns: List[str], base_dir: Path = Path('/kaggle/input')) -> Optional[Path]:
    for pat in patterns:
        matches = list(base_dir.rglob(pat))
        if matches:
            return matches[0]
    return None

def compute_hash(t: str) -> str:
    return hashlib.md5(t.strip().encode('utf-8')).hexdigest()

# 1. Unseen Benchmark Holdout (1,800 samples, 100/class)
bench_path = find_input_file(['clean_benchmark_1800_rows.parquet', '*benchmark*.parquet'])
if bench_path and bench_path.exists():
    df_unseen = pd.read_parquet(bench_path)
    print(f'[Data Source 1] Loaded Unseen Benchmark from {bench_path} (shape: {df_unseen.shape})')
    unseen_hashes = set(df_unseen['text'].map(compute_hash))
else:
    df_unseen = None
    unseen_hashes = set()
    print('[Data Source 1] Unseen benchmark parquet not found.')

# 2. Master Pool V2 (166k rows across 18 classes for Stage 1)
pool_path = find_input_file(['clean_master_pool_v2.parquet', 'clean_master_pool.parquet', '*master*pool*.parquet', '*pool*.parquet'])
if pool_path and pool_path.exists():
    df_pool_raw = pd.read_parquet(pool_path)
    df_pool_raw = df_pool_raw[df_pool_raw['model'].isin(LABELS)].reset_index(drop=True)
    pool_hashes = df_pool_raw['text'].map(compute_hash)
    df_pool = df_pool_raw[~pool_hashes.isin(unseen_hashes)].reset_index(drop=True)
    print(f'[Data Source 2] Loaded Master Pool from {pool_path}: {len(df_pool)} clean passages across {df_pool["model"].nunique()} classes.')
else:
    df_pool = None
    print('[Data Source 2] Master Pool not found; will use train.csv for both stages.')

# 3. Competition train.csv (Prompt-grouped for Stage 2)
train_path = find_input_file(['train.csv'])
if train_path and train_path.exists():
    df_comp_raw = pd.read_csv(train_path)
    df_comp_raw = df_comp_raw[df_comp_raw['model'].isin(LABELS)].reset_index(drop=True)
    comp_hashes = df_comp_raw['text'].map(compute_hash)
    df_comp = df_comp_raw[~comp_hashes.isin(unseen_hashes)].reset_index(drop=True)
    if 'prompt_id' not in df_comp.columns:
        df_comp['prompt_id'] = np.arange(len(df_comp))
    print(f'[Data Source 3] Loaded competition train.csv from {train_path}: {len(df_comp)} prompt-grouped passages.')
else:
    df_comp = None
    print('[Data Source 3] Competition train.csv not found!')

# Fallback validation split if benchmark holdout was not found
if df_unseen is None:
    from sklearn.model_selection import StratifiedGroupKFold
    source_for_split = df_comp if df_comp is not None else df_pool
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    train_idx, val_idx = next(sgkf.split(source_for_split, source_for_split['model'], source_for_split.get('prompt_id', np.arange(len(source_for_split)))))
    df_unseen = source_for_split.iloc[val_idx].reset_index(drop=True)
    if df_comp is not None:
        df_comp = df_comp.iloc[train_idx].reset_index(drop=True)
    print(f'[Data] Generated stratified validation holdout: {len(df_unseen)} samples.')

# Assemble Stage 1 training set: Master Pool balanced sample (or train.csv)
if df_pool is not None and len(df_pool) > 0:
    samples_per_class = CONFIG['stage1']['max_samples'] // K
    sampled_dfs = []
    for model_name in LABELS:
        sub = df_pool[df_pool['model'] == model_name]
        n_take = min(samples_per_class, len(sub))
        sampled_dfs.append(sub.sample(n=n_take, random_state=42))
    df_s1_train = pd.concat(sampled_dfs, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
    if 'prompt_id' not in df_s1_train.columns:
        df_s1_train['prompt_id'] = np.arange(len(df_s1_train))
else:
    df_s1_train = df_comp

# Assemble Stage 2 training set: Prompt-grouped train.csv
df_s2_train = df_comp if df_comp is not None else df_s1_train

print(f"\n{'='*70}")
print(f"📊 Active Training Datasets Assembled:")
print(f"  - Validation Benchmark: {len(df_unseen)} samples (Clean 1,800 Holdout)")
print(f"  - Stage 1 Pretraining: {len(df_s1_train)} samples across {df_s1_train['model'].nunique()} classes (from Master Pool)")
print(f"  - Stage 2 Fine-Tuning: {len(df_s2_train)} samples across {df_s2_train.get('prompt_id', pd.Series()).nunique()} prompts (from train.csv)")
print(f"{'='*70}\n")


In [ ]:
# ==============================================================================
# CELL 4: Tokenizer & PyTorch Datasets
# ==============================================================================
from transformers import AutoTokenizer, AutoModel

# Resolve BGE-M3 base model path (offline dataset or HuggingFace hub)
bge_model_path = CONFIG['backbone']
for cand_p in [Path('/kaggle/input/bge-m3'), Path('/kaggle/input/masterstar-bge-m3'), Path('/kaggle/input/masterstar/bge-m3')]:
    if (cand_p / 'config.json').exists():
        bge_model_path = str(cand_p)
        break
for p in Path('/kaggle/input').rglob('bge-m3'):
    if (p / 'config.json').exists():
        bge_model_path = str(p)
        break

print(f'[Model] Loading BGE-M3 tokenizer from: {bge_model_path}')
tokenizer = AutoTokenizer.from_pretrained(bge_model_path)

class NTLDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, prompt_ids: np.ndarray, tokenizer, max_len: int = 512):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.prompt_ids = np.asarray(prompt_ids)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return {
            'text': str(self.texts[idx]),
            'label': self.labels[idx],
            'prompt_id': self.prompt_ids[idx]
        }

def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    texts = [item['text'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    prompt_ids = [item['prompt_id'] for item in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=CONFIG['max_len'], return_tensors='pt')
    enc['labels'] = labels
    enc['prompt_ids'] = prompt_ids
    return enc

class PromptGroupBatchSampler(Sampler[List[int]]):
    def __init__(self, prompt_ids: np.ndarray, batch_size: int = 4, shuffle: bool = True):
        self.prompt_ids = prompt_ids
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.groups = {}
        for idx, pid in enumerate(prompt_ids):
            self.groups.setdefault(pid, []).append(idx)
        self.group_keys = list(self.groups.keys())

    def __iter__(self) -> Iterator[List[int]]:
        if self.shuffle:
            random.shuffle(self.group_keys)
        batch = []
        for pid in self.group_keys:
            members = list(self.groups[pid])
            if self.shuffle:
                random.shuffle(members)
            for m in members:
                batch.append(m)
                if len(batch) >= self.batch_size:
                    yield batch
                    batch = []
        if batch:
            yield batch

    def __len__(self) -> int:
        return (len(self.prompt_ids) + self.batch_size - 1) // self.batch_size

val_dataset = NTLDataset(
    texts=df_unseen['text'].tolist(),
    labels=df_unseen['model'].map(LABEL2ID).values,
    prompt_ids=df_unseen['prompt_id'].values if 'prompt_id' in df_unseen.columns else np.arange(len(df_unseen)),
    tokenizer=tokenizer,
    max_len=CONFIG['max_len']
)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f'[Data] Validation loader initialized with {len(val_dataset)} samples.')


In [ ]:
# ==============================================================================
# CELL 5: Model Architecture (BGE-M3 + PEFT LoRA + Dual Heads)
# ==============================================================================
from peft import LoraConfig, get_peft_model, TaskType

class BGEClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, hidden_size: int = 1024, num_classes: int = 18):
        super().__init__()
        self.backbone = backbone
        self.hidden_size = hidden_size
        self.num_classes = num_classes

        self.plain_head = nn.Linear(hidden_size, num_classes)
        self.centered_head = nn.Linear(hidden_size, num_classes)
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in (0.1, 0.2, 0.3, 0.4, 0.5)])

    def forward_features(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        cls_repr = outputs.last_hidden_state[:, 0]
        cls_norm = F.normalize(cls_repr, p=2, dim=-1)
        return cls_norm

    def predict_plain(self, Z: torch.Tensor) -> torch.Tensor:
        if self.training:
            return torch.mean(torch.stack([self.plain_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.plain_head(Z)

    def predict_centered(self, Z: torch.Tensor) -> torch.Tensor:
        if self.training:
            return torch.mean(torch.stack([self.centered_head(d(Z)) for d in self.dropouts], dim=0), dim=0)
        return self.centered_head(Z)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        Z = self.forward_features(input_ids, attention_mask)
        logits_plain = self.predict_plain(Z)
        logits_centered = self.predict_centered(Z)
        return Z, logits_plain, logits_centered

print('[Model] Loading base BGE-M3 model into memory...')
base_model = AutoModel.from_pretrained(
    bge_model_path,
    dtype=torch.float32
)

lora_cfg = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=CONFIG['lora_target_modules'],
    bias='none',
    task_type=TaskType.FEATURE_EXTRACTION
)
peft_backbone = get_peft_model(base_model, lora_cfg)
peft_backbone.print_trainable_parameters()

model = BGEClassifier(peft_backbone, hidden_size=CONFIG['hidden_size'], num_classes=K).to(device)
print('[Model] BGEClassifier initialized and transferred to GPU.')


In [ ]:
# ==============================================================================
# CELL 6: Losses & Evaluation Pipeline
# ==============================================================================
family_mask = torch.zeros((K, K), dtype=torch.float32, device=device)
for f_name, members in FAMILIES.items():
    for m1 in members:
        for m2 in members:
            if m1 != m2:
                family_mask[m1, m2] = 1.0

def family_margin_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    family_margin: float = 0.35,
    label_smoothing: float = 0.05
) -> torch.Tensor:
    if family_margin > 0.0:
        margin_matrix = family_mask[labels] * family_margin
        logits = logits + margin_matrix
    return F.cross_entropy(logits, labels, label_smoothing=label_smoothing)

def compute_supcon_loss(features: torch.Tensor, labels: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    norms = F.normalize(features, p=2, dim=-1)
    sim = torch.matmul(norms, norms.T) / temperature
    mask = torch.eq(labels.unsqueeze(1), labels.unsqueeze(0)).float().to(features.device)
    diag_mask = torch.eye(features.shape[0], device=features.device)
    mask = mask * (1.0 - diag_mask)
    pos_per_row = mask.sum(dim=1)
    valid = pos_per_row > 0
    if not valid.any():
        return torch.tensor(0.0, device=features.device, requires_grad=True)
    exp_sim = torch.exp(sim - torch.max(sim, dim=1, keepdim=True)[0]) * (1.0 - diag_mask)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
    mean_log_prob = (mask[valid] * log_prob[valid]).sum(dim=1) / pos_per_row[valid]
    return -mean_log_prob.mean()

@torch.no_grad()
def evaluate_model(model: nn.Module, val_loader: DataLoader) -> Dict[str, float]:
    model.eval()
    all_preds_plain = []
    all_preds_centered = []
    all_probs_blend = []
    all_targets = []

    for batch in val_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()

        with torch.cuda.amp.autocast(dtype=torch.float16):
            Z = model.forward_features(ids, mask)
            lp_p = model.predict_plain(Z)
            lp_c = model.predict_centered(Z)

        p_p = F.softmax(lp_p.float(), dim=-1).cpu().numpy()
        p_c = F.softmax(lp_c.float(), dim=-1).cpu().numpy()
        p_blend = 0.70 * p_p + 0.30 * p_c

        all_preds_plain.append(p_p.argmax(axis=1))
        all_preds_centered.append(p_c.argmax(axis=1))
        all_probs_blend.append(p_blend)
        all_targets.append(labels)

    y_true = np.concatenate(all_targets)
    pred_plain = np.concatenate(all_preds_plain)
    pred_cent = np.concatenate(all_preds_centered)
    prob_blend = np.concatenate(all_probs_blend)
    pred_blend = prob_blend.argmax(axis=1)

    bacc_plain = float(balanced_accuracy_score(y_true, pred_plain))
    bacc_cent = float(balanced_accuracy_score(y_true, pred_cent))
    bacc_blend = float(balanced_accuracy_score(y_true, pred_blend))
    nll = float(log_loss(y_true, prob_blend, labels=np.arange(K)))

    return {
        'bacc_plain': bacc_plain,
        'bacc_cent': bacc_cent,
        'bacc_blend': bacc_blend,
        'nll': nll
    }

print('[Eval] Initial validation baseline on untrained LoRA:')
init_metrics = evaluate_model(model, val_loader)
print(f"  Initial BAcc: Plain={init_metrics['bacc_plain']:.4f} | Centered={init_metrics['bacc_cent']:.4f} | Blend={init_metrics['bacc_blend']:.4f} | NLL={init_metrics['nll']:.4f}")


In [ ]:
# ==============================================================================
# CELL 7: Stage 1 Training (Master Pool Author Discrimination)
# ==============================================================================
from transformers import get_cosine_schedule_with_warmup

train_s1_dataset = NTLDataset(
    texts=df_s1_train['text'].tolist(),
    labels=df_s1_train['model'].map(LABEL2ID).values,
    prompt_ids=df_s1_train['prompt_id'].values,
    tokenizer=tokenizer,
    max_len=CONFIG['max_len']
)
train_s1_loader = DataLoader(
    train_s1_dataset,
    batch_size=CONFIG['stage1']['batch_size'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

optimizer_s1 = torch.optim.AdamW([
    {'params': [p for n, p in model.backbone.named_parameters() if p.requires_grad], 'lr': CONFIG['stage1']['lr_lora']},
    {'params': list(model.plain_head.parameters()) + list(model.centered_head.parameters()), 'lr': CONFIG['stage1']['lr_head']}
], weight_decay=CONFIG['weight_decay'])

total_steps_s1 = (len(train_s1_loader) // CONFIG['stage1']['gradient_accumulation_steps']) * CONFIG['stage1']['epochs']
scheduler_s1 = get_cosine_schedule_with_warmup(optimizer_s1, num_warmup_steps=100, num_training_steps=total_steps_s1)
scaler = torch.cuda.amp.GradScaler()

print(f"[Stage 1] Beginning Stage 1 Pretraining: {len(df_s1_train)} samples | {CONFIG['stage1']['epochs']} epoch | {total_steps_s1} steps")
start_s1 = time.time()

for epoch in range(CONFIG['stage1']['epochs']):
    model.train()
    running_loss = 0.0
    optimizer_s1.zero_grad()

    for step, batch in enumerate(train_s1_loader):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast(dtype=torch.float16):
            Z, logits_p, _ = model(ids, mask)
            loss = family_margin_cross_entropy(
                logits_p,
                labels,
                family_margin=CONFIG['family_margin'],
                label_smoothing=CONFIG['label_smoothing']
            )
            loss = loss / CONFIG['stage1']['gradient_accumulation_steps']

        scaler.scale(loss).backward()
        running_loss += loss.item() * CONFIG['stage1']['gradient_accumulation_steps']

        if (step + 1) % CONFIG['stage1']['gradient_accumulation_steps'] == 0 or (step + 1) == len(train_s1_loader):
            scaler.unscale_(optimizer_s1)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer_s1)
            scaler.update()
            optimizer_s1.zero_grad()
            scheduler_s1.step()

        if (step + 1) % 200 == 0:
            print(f"  [S1 Step {step+1}/{len(train_s1_loader)}] Loss: {running_loss / (step + 1):.4f} | LR: {scheduler_s1.get_last_lr()[0]:.2e}")

    m_s1 = evaluate_model(model, val_loader)
    print(f"[Stage 1 Complete] ({time.time() - start_s1:.1f}s) -> Val BAcc (Blend): {m_s1['bacc_blend']:.4f} | Plain BAcc: {m_s1['bacc_plain']:.4f} | NLL: {m_s1['nll']:.4f}")


In [ ]:
# ==============================================================================
# CELL 8: Stage 2 Training (Prompt-Grouped Centered Twin Head Alignment)
# ==============================================================================
prompt_counts = df_s2_train['prompt_id'].value_counts()
multi_prompts = set(prompt_counts[prompt_counts >= 2].index)
df_s2 = df_s2_train[df_s2_train['prompt_id'].isin(multi_prompts)].reset_index(drop=True)
print(f"[Stage 2] Grouped dataset: {len(df_s2)} rows across {df_s2['prompt_id'].nunique()} multi-sample prompts.")

train_s2_dataset = NTLDataset(
    texts=df_s2['text'].tolist(),
    labels=df_s2['model'].map(LABEL2ID).values,
    prompt_ids=df_s2['prompt_id'].values,
    tokenizer=tokenizer,
    max_len=CONFIG['max_len']
)
batch_sampler_s2 = PromptGroupBatchSampler(df_s2['prompt_id'].values, batch_size=CONFIG['stage2']['batch_size'], shuffle=True)
train_s2_loader = DataLoader(
    train_s2_dataset,
    batch_sampler=batch_sampler_s2,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

optimizer_s2 = torch.optim.AdamW([
    {'params': [p for n, p in model.backbone.named_parameters() if p.requires_grad], 'lr': CONFIG['stage2']['lr_lora']},
    {'params': list(model.plain_head.parameters()) + list(model.centered_head.parameters()), 'lr': CONFIG['stage2']['lr_head']}
], weight_decay=CONFIG['weight_decay'])

total_steps_s2 = (len(train_s2_loader) // CONFIG['stage2']['gradient_accumulation_steps']) * CONFIG['stage2']['epochs']
scheduler_s2 = get_cosine_schedule_with_warmup(optimizer_s2, num_warmup_steps=50, num_training_steps=total_steps_s2)

best_bacc = m_s1['bacc_blend']
val_history = [m_s1]
out_dir = Path(CONFIG['output_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

print(f"[Stage 2] Beginning Contrastive Centered Fine-Tuning: {CONFIG['stage2']['epochs']} epochs | {total_steps_s2} steps")
start_s2 = time.time()

for epoch in range(CONFIG['stage2']['epochs']):
    model.train()
    running_loss = 0.0
    optimizer_s2.zero_grad()

    for step, batch in enumerate(train_s2_loader):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        pids = np.array(batch['prompt_ids'])

        with torch.cuda.amp.autocast(dtype=torch.float16):
            Z = model.forward_features(ids, mask)
            logits_plain = model.predict_plain(Z)
            loss_plain = family_margin_cross_entropy(logits_plain, labels, family_margin=CONFIG['family_margin'], label_smoothing=CONFIG['label_smoothing'])

            # Within-Prompt Centering
            Z_c = Z.clone()
            for p in np.unique(pids):
                m_p = (pids == p)
                if m_p.sum() > 1:
                    Z_c[m_p] = Z_c[m_p] - Z_c[m_p].mean(dim=0, keepdim=True)

            logits_centered = model.predict_centered(Z_c)
            loss_centered = family_margin_cross_entropy(logits_centered, labels, family_margin=CONFIG['family_margin'], label_smoothing=CONFIG['label_smoothing'])
            loss_supcon = compute_supcon_loss(Z_c, labels)

            loss = (loss_plain + CONFIG['stage2']['centered_weight'] * loss_centered + CONFIG['stage2']['supcon_weight'] * loss_supcon) / CONFIG['stage2']['gradient_accumulation_steps']

        scaler.scale(loss).backward()
        running_loss += loss.item() * CONFIG['stage2']['gradient_accumulation_steps']

        if (step + 1) % CONFIG['stage2']['gradient_accumulation_steps'] == 0 or (step + 1) == len(train_s2_loader):
            scaler.unscale_(optimizer_s2)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer_s2)
            scaler.update()
            optimizer_s2.zero_grad()
            scheduler_s2.step()

        if (step + 1) % 200 == 0:
            print(f"  [S2 Epoch {epoch+1} | Step {step+1}/{len(train_s2_loader)}] Loss: {running_loss / (step + 1):.4f} | LR: {scheduler_s2.get_last_lr()[0]:.2e}")

    m_s2 = evaluate_model(model, val_loader)
    val_history.append(m_s2)
    print(f"[Epoch {epoch+1} Complete] Val Blend BAcc: {m_s2['bacc_blend']:.4f} | Plain BAcc: {m_s2['bacc_plain']:.4f} | Centered BAcc: {m_s2['bacc_cent']:.4f} | NLL: {m_s2['nll']:.4f}")

    if m_s2['bacc_blend'] >= best_bacc:
        best_bacc = m_s2['bacc_blend']
        print(f"  🌟 New Best BAcc achieved: {best_bacc:.4f}! Saving checkpoint...")
        model.backbone.save_pretrained(str(out_dir))
        tokenizer.save_pretrained(str(out_dir))
        heads_sd = {
            'plain.weight': model.plain_head.weight.detach().cpu().contiguous(),
            'plain.bias': model.plain_head.bias.detach().cpu().contiguous(),
            'centered.weight': model.centered_head.weight.detach().cpu().contiguous(),
            'centered.bias': model.centered_head.bias.detach().cpu().contiguous(),
        }
        save_file(heads_sd, str(out_dir / 'heads.safetensors'))
        print(f'  Saved adapter and heads to {out_dir}')


In [ ]:
# ==============================================================================
# CELL 9: Final Report & Artifact Export
# ==============================================================================
meta = {
    'model_name': 'bge-m3-lora-inhouse',
    'base_model': CONFIG['backbone'],
    'hidden_size': CONFIG['hidden_size'],
    'num_classes': K,
    'labels': LABELS,
    'families': FAMILIES,
    'config': CONFIG,
    'val_history': val_history,
    'best_val_bacc': best_bacc,
    'completed_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open(out_dir / 'training_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print(f"\n{'='*70}")
print(f"🎉 BGE-M3 LoRA Training Completed Successfully!")
print(f"Best Unseen Benchmark Balanced Accuracy: {best_bacc:.4f}")
print(f"Artifacts exported to: {out_dir}")
print(f"  - adapter_config.json")
print(f"  - adapter_model.safetensors")
print(f"  - heads.safetensors (plain + prompt-centered twin)")
print(f"  - training_meta.json")
print(f"{'='*70}\n")
